In [2]:
%pwd

'c:\\Users\\Aniruddh\\OneDrive\\Desktop\\ai doc\\End-to-end-Medical-Chatbot\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\Aniruddh\\OneDrive\\Desktop\\ai doc\\End-to-end-Medical-Chatbot'

In [ ]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [7]:
def load_pdf_file(data):
    loader=DirectoryLoader(data,
                           glob="*.pdf",
                           loader_cls=PyPDFLoader)
    documents=loader.load()
    return documents

In [8]:
extracted_Data=load_pdf_file("Data")

In [9]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [10]:
text_chunks=text_split(extracted_Data)
print("Length of text chunks:", len(text_chunks))

Length of text chunks: 5859


In [11]:
from langchain.embeddings import HuggingFaceEmbeddings

In [12]:
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [13]:
pip install sentence-transformers


Note: you may need to restart the kernel to use updated packages.


In [14]:
pip uninstall sentence-transformers transformers huggingface_hub -y


Found existing installation: sentence-transformers 5.1.0
Uninstalling sentence-transformers-5.1.0:
  Successfully uninstalled sentence-transformers-5.1.0
Found existing installation: transformers 4.55.2
Uninstalling transformers-4.55.2:
  Successfully uninstalled transformers-4.55.2
Found existing installation: huggingface-hub 0.34.4
Uninstalling huggingface-hub-0.34.4:
  Successfully uninstalled huggingface-hub-0.34.4
Note: you may need to restart the kernel to use updated packages.


In [15]:
pip install sentence-transformers


  Using cached sentence_transformers-5.1.0-py3-none-any.whl.metadata (16 kB)
  Using cached huggingface_hub-0.34.4-py3-none-any.whl.metadata (14 kB)
Using cached sentence_transformers-5.1.0-py3-none-any.whl (483 kB)
   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
   -- ------------------------------------- 0.8/11.3 MB 3.1 MB/s eta 0:00:04
   ----- ---------------------------------- 1.6/11.3 MB 3.5 MB/s eta 0:00:03
   --------- ------------------------------ 2.6/11.3 MB 3.7 MB/s eta 0:00:03
   ----------- ---------------------------- 3.1/11.3 MB 3.5 MB/s eta 0:00:03
   ------------- -------------------------- 3.9/11.3 MB 3.5 MB/s eta 0:00:03
   --------------- ------------------------ 4.5/11.3 MB 3.5 MB/s eta 0:00:02
   ------------------ --------------------- 5.2/11.3 MB 3.3 MB/s eta 0:00:02
   -------------------- ------------------- 5.8/11.3 MB 3.2 MB/s eta 0:00:02
   --------------------- ---

In [16]:
embeddings = download_hugging_face_embeddings()

C:\Users\Aniruddh\AppData\Local\Temp\ipykernel_4804\789423792.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
c:\Users\Aniruddh\OneDrive\Desktop\ai doc\End-to-end-Medical-Chatbot\medibot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
query_result = embeddings.embed_query("Hi andy")
print("Length",len(query_result))

Length 384


In [18]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
PINECONE_API_KEY = os.getenv("")
GROQ_API_KEY = os.getenv("")

In [ ]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key=)

index_name = "medicalchatbot"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "medicalchatbot",
    "metric": "cosine",
    "host": "medicalchatbot-8cmigah.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [ ]:
#import os 
#os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
#os.environ["GROQ_API_KEY"] = GROQ_API_KEY


In [22]:
pip install langchain-pinecone


Note: you may need to restart the kernel to use updated packages.


In [23]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings
)

In [24]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)


In [25]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [26]:
pip install langchain-groq


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # load .env file

# now your code can read keys:
print("Pinecone:", os.getenv("))
print("Groq:", os.getenv(""))


Pinecone: pcsk_6bmPm1_Qt29YngGHr5TXp3DYSZyPLETJBhGsx8C4nbEq7kqgNGS6BaEiTtXhAfoP7HhVxF
Groq: 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


In [28]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0.4,
    max_tokens=500
)


In [29]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [30]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [31]:
response = rag_chain.invoke({"input": "what is Acne"})
print(response["answer"])

Acne is a common skin disease characterized by pimples on the face, chest, and back. It occurs when the pores of the skin become clogged with oil, dead skin cells, and bacteria. Acne vulgaris, the medical term for common acne, is the most common skin disease, affecting nearly 17 million people in the United States.
